# 🚀 AI-Powered Academic QA: NLP Pipeline Analysis

**Objective:** Test individual components (Extraction, Chunking, Embedding) and the full pipeline dynamically across all files in `sample_files/`.

**Key Features:**
- **Dynamic Discovery:** Automatically finds `.pdf`, `.docx`, `.txt`.
- **Persistent Logging:** Saves performance metrics to JSON for cross-run comparison.
- **Robust Error Handling:** Isolates failures so one bad file doesn't crash the whole suite.
- **Mentor Comments:** Explains the *What, Why, Where, and How* of each step.

In [20]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS (DEBUGGED)
# ==============================================================================
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. Load Environment Variables
PROJECT_ROOT = Path.cwd().parent.parent
env_path = PROJECT_ROOT / ".env"

if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True) # override=True ensures it updates os.environ
    print(f"✅ Loaded .env from: {env_path}")
else:
    print(f"❌ .env not found at: {env_path}")

# 2. DEBUG: Check if variables are actually in os.environ
print("\n🔍 Checking os.environ:")
print(f"   SUPABASE_URL present: {'SUPABASE_URL' in os.environ}")
print(f"   SUPABASE_KEY present: {'SUPABASE_KEY' in os.environ}")

if 'SUPABASE_URL' not in os.environ:
    print("   ⚠️ Variables missing! Check your .env file format.")
    print("   Example: SUPABASE_URL=https://...")

# 3. Add Project Root to Path
sys.path.insert(0, str(PROJECT_ROOT))

# 4. Import App Modules
try:
    from app.nlp.extract import extract_document
    from app.nlp.chunk import chunk_pages
    from app.nlp.embed import embed_texts, get_model
    from app.logging_system import log_extraction, log_chunking, log_embedding
    from app.config import settings
    print("✅ App modules imported successfully.")
except Exception as e:
    print(f"❌ Configuration Error: {e}")

✅ Loaded .env from: /home/kenzo/Documents/KaisyCode/project/DocuChat/backend/.env

🔍 Checking os.environ:
   SUPABASE_URL present: True
   SUPABASE_KEY present: True


/home/kenzo/Documents/KaisyCode/project/DocuChat/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ App modules imported successfully.


In [ ]:
# ==============================================================================
# CELL 2: HELPER FUNCTIONS & LOGGING UTILITIES
# ==============================================================================
# WHAT: Utility functions to handle file discovery and persistent logging.
# WHY: To keep the main logic clean and ensure we can compare results later.
# HOW: Using pathlib for OS-agnostic file handling and json for structured logs.

import json
from datetime import datetime
from typing import List, Dict, Any, Optional
import numpy as np


def discover_files(directory: Path) -> List[Path]:
    """Dynamically find all supported files in the directory."""
    if not directory.exists():
        return []
    files = []
    for ext in SUPPORTED_EXTENSIONS:
        files.extend(directory.glob(f"*{ext}"))
    return sorted(files)

def get_file_type(file_path: Path) -> Optional[str]:
    """Return file type string based on extension."""
    ext = file_path.suffix.lower()
    return ext[1:] if ext in SUPPORTED_EXTENSIONS else None

def save_run_log(run_id: str, data: Dict[str, Any]):
    """
    Persist test results to a JSON file.
    WHY: Allows us to compare performance across different runs or code changes.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = LOG_DIR / f"run_{run_id}_{timestamp}.json"
    
    # Convert numpy types to native Python types for JSON serialization
    def convert_numpy(obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return obj

    with open(log_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, default=convert_numpy)
    
    return log_file

In [18]:
# ==============================================================================
# CELL 3: COMPONENT TEST - TEXT EXTRACTION
# ==============================================================================
# WHAT: Tests the ability to pull raw text from various academic formats.
# WHY: Garbage In = Garbage Out. If extraction fails, the QA system fails.
# HOW: Iterates through discovered files, measures time, and logs character counts.
import time
print("\n🔍 Starting Extraction Tests...")
files = discover_files(SAMPLE_DIR)
extraction_results = []

if not files:
    print("⚠️ No sample files found. Please add .pdf/.docx/.txt to sample_files/")
else:
    for file_path in files:
        file_type = get_file_type(file_path)
        if not file_type: continue

        print(f"\n📄 Processing: {file_path.name}")
        try:
            start_time = time.time()
            with open(file_path, 'rb') as f:
                content = f.read()
            
            pages = extract_document(content, file_type)
            duration = time.time() - start_time
            
            total_chars = sum(len(p.get('text', '')) for p in pages)
            
            result = {
                "file": file_path.name,
                "type": file_type,
                "status": "success",
                "pages": len(pages),
                "total_chars": total_chars,
                "duration_sec": round(duration, 4)
            }
            extraction_results.append(result)
            print(f"   ✅ Extracted {len(pages)} pages ({total_chars} chars) in {duration:.2f}s")

        except Exception as e:
            print(f"   ❌ Failed: {str(e)}")
            extraction_results.append({
                "file": file_path.name,
                "status": "error",
                "error": str(e)
            })

# Save Extraction Logs
log_path = save_run_log("extraction", {"results": extraction_results})
print(f"\n💾 Extraction logs saved to: {log_path}")


🔍 Starting Extraction Tests...

📄 Processing: 5th sem.pdf
   ✅ Extracted 19 pages (38958 chars) in 6.13s

📄 Processing: 6th sem.pdf
   ✅ Extracted 24 pages (43316 chars) in 6.17s

📄 Processing: sample.txt
   ❌ Failed: Unsupported file type: txt

💾 Extraction logs saved to: /home/kenzo/Documents/KaisyCode/project/DocuChat/backend/tests/nlp_tests/logs/run_extraction_20261006_222322.json


In [16]:
# ==============================================================================
# CELL 4: COMPONENT TEST - SEMANTIC CHUNKING
# ==============================================================================
# WHAT: Breaks extracted text into manageable, semantically coherent pieces.
# WHY: LLMs have context windows. Chunking ensures we feed relevant context.
# HOW: Uses the chunk_pages function and analyzes chunk distribution.

print("\n🔪 Starting Chunking Tests...")
chunking_results = []

# We re-extract here to ensure we are testing chunking on fresh data
# In a real pipeline, this would come from a database/cache
for file_info in extraction_results:
    if file_info.get('status') != 'success': continue
    
    file_name = file_info['file']
    file_type = file_info['type']
    file_path = SAMPLE_DIR / file_name
    
    try:
        with open(file_path, 'rb') as f:
            pages = extract_document(f.read(), file_type)
        
        start_time = time.time()
        chunks = chunk_pages(pages)
        duration = time.time() - start_time
        
        avg_size = np.mean([len(c['text']) for c in chunks]) if chunks else 0
        
        result = {
            "file": file_name,
            "status": "success",
            "chunk_count": len(chunks),
            "avg_chunk_size": round(avg_size, 2),
            "duration_sec": round(duration, 4)
        }
        chunking_results.append(result)
        print(f"   ✅ {file_name}: {len(chunks)} chunks (Avg: {avg_size:.0f} chars) in {duration:.2f}s")

    except Exception as e:
        print(f"   ❌ Chunking failed for {file_name}: {e}")
        chunking_results.append({"file": file_name, "status": "error", "error": str(e)})

log_path = save_run_log("chunking", {"results": chunking_results})
print(f"\n💾 Chunking logs saved to: {log_path}")


🔪 Starting Chunking Tests...
   ✅ 5th sem.pdf: 93 chunks (Avg: 432 chars) in 0.00s
   ✅ 6th sem.pdf: 103 chunks (Avg: 433 chars) in 0.00s

💾 Chunking logs saved to: /home/kenzo/Documents/KaisyCode/project/DocuChat/backend/tests/nlp_tests/logs/run_chunking_20261006_213953.json


In [22]:
# ==============================================================================
# CELL 5: COMPONENT TEST - VECTOR EMBEDDINGS (FIXED & LOCAL CACHE)
# ==============================================================================
# WHAT: Converts text chunks into numerical vectors for semantic search.
# WHY: Core of the "AI" part. Allows finding answers by meaning.
# HOW: 
#   1. Sets a local cache path to avoid re-downloading.
#   2. Loads the model once.
#   3. Correctly accesses 'sentence_embedding_dimension' from the model object.

import os
from sentence_transformers import SentenceTransformer

print("\n🔄 Starting Embedding Tests...")
embedding_results = []

# 1. Define Local Cache Directory
#    This ensures the model is downloaded once and stored in your project folder.
LOCAL_CACHE_DIR = PROJECT_ROOT / "models" / "sentence_transformers"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

try:
    # 2. Load Model with Local Cache
    model_name = settings.EMBEDDING_MODEL
    print(f"   ⏳ Loading model: {model_name}")
    print(f"   📂 Cache location: {LOCAL_CACHE_DIR}")
    
    # sentence-transformers automatically checks this folder before hitting HuggingFace
    model = SentenceTransformer(model_name, cache_folder=str(LOCAL_CACHE_DIR))
    
    # 3. Correctly Get Dimension
    #    The dimension is a property of the main model object, not the pooling layer.
    model_dim = model.get_sentence_embedding_dimension()
    
    if model_dim is None:
        raise ValueError("Could not determine embedding dimension. Check model configuration.")
        
    print(f"   ✅ Model loaded successfully (Dim: {model_dim})")

    # 4. Process Files
    for file_info in extraction_results:
        if file_info.get('status') != 'success': continue
        
        file_name = file_info['file']
        file_type = file_info['type']
        file_path = SAMPLE_DIR / file_name

        try:
            # Re-extract and chunk for isolation
            with open(file_path, 'rb') as f:
                pages = extract_document(f.read(), file_type)
            chunks = chunk_pages(pages)
            texts = [c['text'] for c in chunks]

            if not texts:
                print(f"   ⚠️ Skipping {file_name}: No text to embed.")
                continue

            start_time = time.time()
            # Use the model directly to encode
            embeddings = model.encode(texts, show_progress_bar=False)
            duration = time.time() - start_time

            # Convert to list for JSON serialization if needed, but keep as np for stats
            emb_array = np.array(embeddings)
            norms = np.linalg.norm(emb_array, axis=1)
            
            result = {
                "file": file_name,
                "status": "success",
                "vector_count": len(embeddings),
                "dimension": int(model_dim),
                "mean_norm": round(float(np.mean(norms)), 4),
                "duration_sec": round(duration, 4)
            }
            embedding_results.append(result)
            print(f"   ✅ {file_name}: {len(embeddings)} vectors in {duration:.2f}s")

        except Exception as e:
            print(f"   ❌ Embedding failed for {file_name}: {e}")
            embedding_results.append({"file": file_name, "status": "error", "error": str(e)})

except Exception as e:
    print(f"❌ Critical Model Error: {e}")
    import traceback
    traceback.print_exc()

# Save Logs
if embedding_results:
    log_path = save_run_log("embedding", {"results": embedding_results})
    print(f"\n💾 Embedding logs saved to: {log_path}")
else:
    print("\n⚠️ No embedding results to save.")

2026-10-06 22:41:40,457 - sentence_transformers.base.model - INFO - No device provided, using cuda:0



🔄 Starting Embedding Tests...
   ⏳ Loading model: all-MiniLM-L6-v2
   📂 Cache location: /home/kenzo/Documents/KaisyCode/project/DocuChat/backend/models/sentence_transformers


2026-10-06 22:41:40,968 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 22:41:41,038 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-06 22:41:41,111 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-06 22:41:41,388 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/r

   ✅ Model loaded successfully (Dim: 384)
   ✅ 5th sem.pdf: 93 vectors in 1.39s
   ✅ 6th sem.pdf: 103 vectors in 0.43s

💾 Embedding logs saved to: /home/kenzo/Documents/KaisyCode/project/DocuChat/backend/tests/nlp_tests/logs/run_embedding_20261006_224216.json


In [24]:
# ==============================================================================
# CELL 6: FULL PIPELINE INTEGRATION TEST
# ==============================================================================
# WHAT: Runs the entire flow: Extract -> Chunk -> Embed.
# WHY: To identify bottlenecks and ensure end-to-end compatibility.
# HOW: Measures total time and breaks down time per stage.

print("\n🚀 Starting Full Pipeline Integration Test...")
pipeline_results = []

for file_path in discover_files(SAMPLE_DIR):
    file_type = get_file_type(file_path)
    if not file_type: continue

    print(f"\n🔄 Pipeline: {file_path.name}")
    timings = {}
    
    try:
        # Stage 1: Extraction
        t0 = time.time()
        with open(file_path, 'rb') as f:
            pages = extract_document(f.read(), file_type)
        timings['extraction'] = time.time() - t0
        
        if not pages: raise ValueError("No text extracted")

        # Stage 2: Chunking
        t1 = time.time()
        chunks = chunk_pages(pages)
        timings['chunking'] = time.time() - t1
        
        if not chunks: raise ValueError("No chunks created")

        # Stage 3: Embedding
        t2 = time.time()
        texts = [c['text'] for c in chunks]
        embeddings = embed_texts(texts)
        timings['embedding'] = time.time() - t2

        total_time = sum(timings.values())
        
        result = {
            "file": file_path.name,
            "status": "success",
            "timings": {k: round(v, 4) for k, v in timings.items()},
            "total_time_sec": round(total_time, 4),
            "stats": {
                "pages": len(pages),
                "chunks": len(chunks),
                "vectors": len(embeddings)
            }
        }
        pipeline_results.append(result)
        
        # Print breakdown
        print(f"   ⏱️ Total: {total_time:.2f}s")
        for stage, t in timings.items():
            pct = (t / total_time) * 100
            print(f"      - {stage.capitalize():<12}: {t:.2f}s ({pct:.1f}%)")

    except Exception as e:
        print(f"   ❌ Pipeline failed: {e}")
        pipeline_results.append({"file": file_path.name, "status": "error", "error": str(e)})

log_path = save_run_log("full_pipeline", {"results": pipeline_results})
print(f"\n💾 Pipeline logs saved to: {log_path}")


🚀 Starting Full Pipeline Integration Test...

🔄 Pipeline: 5th sem.pdf
   ⏱️ Total: 6.59s
      - Extraction  : 6.22s (94.3%)
      - Chunking    : 0.00s (0.0%)
      - Embedding   : 0.38s (5.7%)

🔄 Pipeline: 6th sem.pdf
   ⏱️ Total: 7.50s
      - Extraction  : 7.09s (94.6%)
      - Chunking    : 0.00s (0.0%)
      - Embedding   : 0.41s (5.4%)

💾 Pipeline logs saved to: /home/kenzo/Documents/KaisyCode/project/DocuChat/backend/tests/nlp_tests/logs/run_full_pipeline_20261006_224545.json


In [25]:
# ==============================================================================
# CELL 7: FINAL SUMMARY & ANALYSIS
# ==============================================================================
# WHAT: Aggregates all results into a readable summary.
# WHY: To quickly identify which files are problematic or which stages are slow.

print("\n" + "="*60)
print("📊 FINAL TEST SUMMARY")
print("="*60)

all_results = {
    "Extraction": extraction_results,
    "Chunking": chunking_results,
    "Embedding": embedding_results,
    "Full Pipeline": pipeline_results
}

for stage, results in all_results.items():
    success_count = sum(1 for r in results if r.get('status') == 'success')
    total_count = len(results)
    status_icon = "✅" if success_count == total_count else "⚠️"
    print(f"{status_icon} {stage:<15}: {success_count}/{total_count} successful")

print("\n💡 Mentor Tip: Check the 'logs/' folder for detailed JSON files.")
print("   Use these files to compare performance if you change models or chunking strategies.")
print("="*60)


📊 FINAL TEST SUMMARY
⚠️ Extraction     : 2/3 successful
✅ Chunking       : 2/2 successful
✅ Embedding      : 2/2 successful
✅ Full Pipeline  : 2/2 successful

💡 Mentor Tip: Check the 'logs/' folder for detailed JSON files.
   Use these files to compare performance if you change models or chunking strategies.
